In [1]:
import requests
import pandas as pd
import time
from io import BytesIO
from datetime import datetime

import sys
sys.path.append('..')
from utils.bucket_utils import get_duck_con

In [2]:
import os
!{sys.executable} -m pip install playwright pandas pyarrow fastparquet
!{sys.executable} -m playwright install chromium

  Using cached playwright-1.58.0-py3-none-macosx_11_0_arm64.whl (41.0 MB)
  Using cached pyarrow-21.0.0-cp39-cp39-macosx_12_0_arm64.whl (31.2 MB)
  Using cached fastparquet-2024.11.0-cp39-cp39-macosx_11_0_arm64.whl (684 kB)
  Using cached greenlet-3.2.5-cp39-cp39-macosx_11_0_universal2.whl (274 kB)
  Using cached pyee-13.0.1-py3-none-any.whl (15 kB)
  Using cached cramjam-2.11.0-cp39-cp39-macosx_11_0_arm64.whl (1.7 MB)
You should consider upgrading via the '/Users/palmchns/113-pea-oms/.venv/bin/python -m pip install --upgrade pip' command.


In [ ]:
import asyncio
from playwright.async_api import async_playwright, TimeoutError
import pandas as pd
import os
from datetime import datetime

In [ ]:
def get_today_thai_format():
    now = datetime.now()
    thai_months = ["มกราคม", "กุมภาพันธ์", "มีนาคม", "เมษายน", "พฤษภาคม", "มิถุนายน",
                   "กรกฎาคม", "สิงหาคม", "กันยายน", "ตุลาคม", "พฤศจิกายน", "ธันวาคม"]
    thai_year = now.year + 543
    return f"{now.day:02d} {thai_months[now.month - 1]} {thai_year}"

async def scrape_pathum_all_districts():
    keywords = [
        "เมืองปทุมธานี", "คลองหลวง", "ธัญบุรี", "ลำลูกกา", "สามโคก", "ลาดหลุมแก้ว", "หนองเสือ", "รังสิต",
        "ถนนกาญจนาภิเษก", "ถนนพหลโยธิน", "อุดรรัถยา", "ถนนรังสิตนครนายก", "ติวานนท์", "รังสิตปทุมธานี", 
        "เชียงราก", "เลียบคลองเปรมประชากร", "ไสวประชาราษฎร์", "คลอง 1", "คลอง 2", "คลอง 3", "คลอง 4", "คลอง 5"
    ]
    all_data = []
    today_real_date = get_today_thai_format() 

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, args=["--disable-blink-features=AutomationControlled"])
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
        page = await context.new_page()

        for keyword in keywords:
            print(f"🔍 กำลังดึงข้อมูลพื้นที่: {keyword}...")
            try:
                await page.goto("https://www.js100.com/en/site/home/search_advance", wait_until="domcontentloaded", timeout=30000)
                await page.wait_for_timeout(1000)
                
                search_input = await page.wait_for_selector('input[name="search_text"], #search_result_input', timeout=10000)
                await search_input.click()
                await search_input.fill(keyword)
                await page.wait_for_timeout(500)
                await search_input.press("Enter")
                
                await page.wait_for_selector('#search_result_list li', timeout=20000)
                await page.wait_for_timeout(1000)
                
                items = await page.query_selector_all('#search_result_list li')
                for item in items:
                    h4_tag = await item.query_selector('h4')
                    raw_datetime = (await h4_tag.inner_text()).strip() if h4_tag else ""
                    date_str, time_str = "ไม่ระบุ", "ไม่ระบุ"
                    if "," in raw_datetime:
                        parts = raw_datetime.split(",")
                        date_str, time_str = parts[0].strip(), parts[1].strip()
                    else:
                        date_str = raw_datetime
                    
                    if date_str == "วันนี้": 
                        date_str = today_real_date
                    
                    a_tag, p_tag = await item.query_selector('a'), await item.query_selector('p')
                    category, title, details = "", "", ""
                    if a_tag and (await a_tag.inner_text()).strip():
                        category, title = "ข่าว/ประกาศ", (await a_tag.inner_text()).strip()
                        link = await a_tag.get_attribute('href')
                        details = f"https://www.js100.com{link}" if link and link.startswith('/') else link
                    elif p_tag:
                        category, title, details = "รายงานจราจร", f"อัปเดตจราจรพื้นที่ {keyword}", (await p_tag.inner_text()).strip()
                    else:
                        category, title, details = "อื่นๆ", "ไม่มีหัวข้อ", (await item.inner_text()).replace(raw_datetime, "").strip()

                    all_data.append({"พื้นที่ (คำค้น)": keyword, "วันที่": date_str, "เวลา": time_str, "ประเภท": category, "หัวข้อ": title, "รายละเอียด": details})
            
            except TimeoutError:
                print(f"   ⚠️ ข้ามพื้นที่ '{keyword}' (ใช้เวลาโหลดนานเกินไป หรือไม่พบข้อมูล)")
                continue
            except asyncio.CancelledError:
                print(f"\n🛑 การทำงานถูกยกเลิกกลางคัน! (อาจเผลอปิดเบราว์เซอร์ หรือกดหยุด)")
                break
            except Exception as e:
                print(f"   ❌ เกิดข้อผิดพลาดกับ '{keyword}': {e}")
                continue

        await browser.close()
    return pd.DataFrame(all_data)

In [ ]:
async def main():
    print("🚀 เริ่มกระบวนการดึงข้อมูล...")
    df_new = await scrape_pathum_all_districts()

    # 1. เตรียมข้อมูลวันที่และเวลาสำหรับชื่อไฟล์
    now = datetime.now()
    today_real_date = get_today_thai_format() # "26 มีนาคม 2569"
    timestamp = now.strftime("%Y%m%d_%H%M")   # "20260326_1740"

    # 2. กำหนด Path โฟลเดอร์
    desktop_path = os.path.expanduser("~/Desktop")
    target_folder = os.path.join(desktop_path, "JS100")

    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print(f"📁 สร้างโฟลเดอร์ใหม่ที่: {target_folder}")

    # 3. จัดการข้อมูล
    if not df_new.empty:
        # กรองเอาเฉพาะของวันนี้
        df_today = df_new[df_new['วันที่'] == today_real_date].copy()
        
        if df_today.empty:
            print(f"⚠️ ไม่พบรายการของวันที่ {today_real_date} (ข้ามการบันทึก)")
        else:
            file_name = os.path.join(target_folder, f"JS100_Pathum_{timestamp}.parquet")
            
            # บันทึกไฟล์
            df_today.to_parquet(file_name, engine='fastparquet', index=False)
            
            print(f"✅ บันทึกข้อมูลสำเร็จ!")
            print(f"📄 ชื่อไฟล์: JS100_Pathum_{timestamp}.parquet")
            print(f"📍 ที่อยู่: {file_name}")
            
            # 💡 เปลี่ยนจาก display เป็น print เพื่อไม่ให้ Error นอก Jupyter
            print(df_today)
    else:
        print("🤷‍♂️ ไม่พบข้อมูลใหม่จากการค้นหาครั้งนี้")

    print("✅ กระบวนการเสร็จสิ้น!")

# --- จุดสตาร์ทของระบบ ---
if __name__ == "__main__":
    table_today = await main()

In [ ]:
async def main():
    print("🚀 เริ่มกระบวนการดึงข้อมูล...")
    df_new = await scrape_pathum_all_districts()

    now = datetime.now()
    today_real_date = get_today_thai_format() 
    timestamp = now.strftime("%Y%m%d_%H%M")   

    desktop_path = os.path.expanduser("~/Desktop")
    target_folder = os.path.join(desktop_path, "JS100")

    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print(f"📁 สร้างโฟลเดอร์ใหม่ที่: {target_folder}")

    if not df_new.empty:
        # 1. กรองเอาเฉพาะของวันนี้
        df_today = df_new[df_new['วันที่'] == today_real_date].copy()
        
        # 2. 💡 ลบข้อมูลที่ซ้ำซ้อนกัน (เช็คจาก วันที่, เวลา, หัวข้อ และรายละเอียด)
        # ใช้ keep='first' เพื่อเก็บข้อมูลที่ค้นเจอครั้งแรกไว้ และลบตัวที่ซ้ำออก
        df_today = df_today.drop_duplicates(subset=['วันที่', 'เวลา', 'หัวข้อ', 'รายละเอียด'], keep='first')
        
        if df_today.empty:
            print(f"⚠️ ไม่พบรายการของวันที่ {today_real_date} (ข้ามการบันทึก)")
            return pd.DataFrame()
        else:
            file_name = os.path.join(target_folder, f"JS100_Pathum_{timestamp}.parquet")
            
            df_today.to_parquet(file_name, engine='fastparquet', index=False)
            
            print(f"✅ บันทึกข้อมูลสำเร็จ! (กรองข้อมูลซ้ำออกแล้ว)")
            print(f"📄 ชื่อไฟล์: JS100_Pathum_{timestamp}.parquet")
            print(f"📍 ที่อยู่: {file_name}")
            print("✅ กระบวนการเสร็จสิ้น!")
            
            # ส่งตารางออกมาให้ใช้งานต่อ
            return df_today
    else:
        print("🤷‍♂️ ไม่พบข้อมูลใหม่จากการค้นหาครั้งนี้")
        return pd.DataFrame()

if __name__ == "__main__":
    # บันทึกข้อมูลที่ได้ลงตัวแปร table_today
    table_today = await main()
    
    # แสดงผลตาราง (ถ้ามีข้อมูล)
    if not table_today.empty:
        display(table_today)

In [ ]:
display(table_today)